# EmoText MZ — Detecção de Emoções (AfriEmo PT-MZ) — configuração corrigida

**PFC:** Ivany Mazia Massinga — LECC51/5976 — ISUTC

## Enquadramento
Neste trabalho, a análise emocional de textos digitais compreende **dois níveis complementares**:

| Nível | Modelo teórico | Dados | Estado |
|---|---|---|---|
| 1. Polaridade (valência): positivo, negativo, neutro | Modelo dimensional (Russell 1980) | AfriSenti — PT-MZ e Xitsonga | concluído na 1.ª execução |
| 2. Emoções discretas: alegria, tristeza, raiva, medo, surpresa, nojo | Modelo categórico (Ekman 1992) | AfriEmo — PT-MZ | **este notebook** |

A análise incide exclusivamente sobre **texto**, em contexto **multilíngue** (português de Moçambique e xitsonga).

## Porque se repete o nível 2
Na primeira execução, os modelos Transformer convergiram para a previsão sistemática da **ausência de emoções**
(XLM-R com Macro-F1 = 0), comportamento típico em classificação multi-rótulo com classes positivas raras
(4–18% dos textos) e limiar fixo de 0,5. Este notebook repete **apenas a detecção de emoções** com:

1. **Perda ponderada**: cada emoção recebe peso positivo √(negativos/positivos), calculado no treino;
2. **10 épocas** e taxa de aprendizagem **3e-5**, com selecção da melhor época por Macro-F1 na validação;
3. **Limiar de decisão calibrado por emoção no conjunto de validação** (nunca no teste);
4. O baseline TF-IDF + Regressão Logística recebe a mesma calibração de limiares, para uma comparação justa.





## 1. Configuração

In [ ]:
import os
import sys
import subprocess

EM_COLAB = 'google.colab' in sys.modules
TESTE_RAPIDO = os.environ.get('EMOTEXT_TESTE_RAPIDO', '0') == '1'

if EM_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate>=1.0'], check=True)
    from google.colab import drive


    DRIVE_LIGADO = False
    for tentativa in (1, 2):
        try:
            drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
            DRIVE_LIGADO = True
            break
        except Exception as erro:
            print(f'AVISO: falha ao ligar o Google Drive (tentativa {tentativa}/2): {erro}')

    PASTA_BASE = None
    if DRIVE_LIGADO:
        PASTA_BASE = '/content/drive/MyDrive/EmoTextColab'
        if not os.path.isdir(PASTA_BASE):
            candidatos = [os.path.join(r, d) for r, ds, _ in os.walk('/content/drive/MyDrive') for d in ds if d == 'EmoTextColab']
            PASTA_BASE = candidatos[0] if candidatos else None
            if PASTA_BASE is None:
                print('AVISO: pasta EmoTextColab não encontrada no Drive.')


    if PASTA_BASE is None:
        PASTA_BASE = '/content/EmoTextColab'
        print('O notebook continua SEM o Google Drive: os resultados ficam em /content e o zip é descarregado no fim.')
        print('Mantenha o separador aberto até ao fim (sem o Drive não há retoma se a sessão cair).')
    PASTA_TEMP = '/content/tmp_treino'

    pasta_dados = os.path.join(PASTA_BASE, 'dados')
    if not all(os.path.exists(os.path.join(pasta_dados, f'afriemo_ptmz_{s}.csv')) for s in ['train', 'dev', 'test']):
        try:
            from datasets import load_dataset
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
            from datasets import load_dataset
        os.makedirs(pasta_dados, exist_ok=True)
        brighter = load_dataset('brighter-dataset/BRIGHTER-emotion-categories', 'ptmz')
        for s in ['train', 'dev', 'test']:
            brighter[s].to_pandas().to_csv(os.path.join(pasta_dados, f'afriemo_ptmz_{s}.csv'), index=False, encoding='utf-8')
        print('Dados AfriEmo PT-MZ descarregados do Hugging Face (brighter-dataset/BRIGHTER-emotion-categories, ptmz).')
else:
    PASTA_BASE = os.environ.get('EMOTEXT_BASE', os.getcwd())
    PASTA_TEMP = os.path.join(PASTA_BASE, '_tmp_treino')

PASTA_DADOS = os.path.join(PASTA_BASE, 'dados')
PASTA_RES = os.path.join(PASTA_BASE, 'resultados_teste' if TESTE_RAPIDO else 'resultados_emocoes_v2')
SUB = {k: os.path.join(PASTA_RES, k) for k in ['logs', 'execucoes', 'previsoes', 'relatorios', 'tabelas', 'figuras', 'anexos']}
for p in list(SUB.values()) + [PASTA_TEMP]:
    os.makedirs(p, exist_ok=True)
print('Pasta base:', PASTA_BASE)
print('Resultados:', PASTA_RES)

Mounted at /content/drive
Pasta base: /content/drive/MyDrive/EmoTextColab
Resultados: /content/drive/MyDrive/EmoTextColab/resultados_emocoes_v2


In [ ]:
import glob
import io
import json
import logging
import re
import shutil
import time
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import transformers
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import accuracy_score, f1_score, classification_report, multilabel_confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, TrainerCallback, set_seed)

SEMENTES = [42, 43, 44]
MODELOS = {
    'mBERT': 'google-bert/bert-base-multilingual-cased',
    'XLM-R': 'FacebookAI/xlm-roberta-base',
    'AfroXLMR': 'Davlan/afro-xlmr-base',
}
HIPER = {
    'max_comprimento': 128,
    'taxa_aprendizagem': 3e-5,
    'tamanho_lote': 16,
    'epocas': 10,
    'decaimento_pesos': 0.01,
    'proporcao_aquecimento': 0.1,
    'peso_positivo': 'raiz quadrada de (negativos / positivos) por emoção, calculada no treino',
    'limiares': 'calibrados por emoção na validação (grelha 0,05 a 0,95, passo 0,05)',
    'selecao_modelo': 'melhor época por Macro-F1 na validação (com limiares calibrados)',
}
EMOCOES = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
EMO_PT = {'anger': 'Raiva', 'disgust': 'Nojo', 'fear': 'Medo', 'joy': 'Alegria', 'sadness': 'Tristeza', 'surprise': 'Surpresa'}
GRELHA_LIMIARES = np.round(np.arange(0.05, 0.951, 0.05), 2)
URL_NRC = 'http://saifmohammad.com/WebDocs/Lexicons/NRC-Emotion-Lexicon.zip'

FICHEIRO = {'Maioritária': 'Maioritaria', 'Léxico NRC': 'Lexico_NRC', 'TF-IDF+RL': 'TFIDF_RL',
            'mBERT': 'mBERT', 'XLM-R': 'XLM-R', 'AfroXLMR': 'AfroXLMR'}
BASELINES = ('Maioritária', 'Léxico NRC', 'TF-IDF+RL')
ORDEM_MODELOS = list(BASELINES) + list(MODELOS)

if TESTE_RAPIDO:
    SEMENTES = [42, 43]
    MODELOS = {'XLM-R': 'FacebookAI/xlm-roberta-base'}
    HIPER['epocas'] = 1

## 2. Registo de execução (log) e ambiente

In [ ]:
CAMINHO_LOG = os.path.join(SUB['logs'], f'execucao_{datetime.now():%Y%m%d_%H%M%S}.log')
log = logging.getLogger('emotext')
log.setLevel(logging.INFO)
log.handlers.clear()
for h in [logging.FileHandler(CAMINHO_LOG, encoding='utf-8'), logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%Y-%m-%d %H:%M:%S'))
    log.addHandler(h)
for nome_ruidoso in ['transformers', 'datasets', 'urllib3', 'filelock']:
    logging.getLogger(nome_ruidoso).setLevel(logging.ERROR)
transformers.utils.logging.set_verbosity_error()

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
ambiente = {
    'data_execucao': f'{datetime.now():%Y-%m-%d %H:%M:%S}',
    'plataforma': 'Google Colab' if EM_COLAB else 'Local',
    'gpu': torch.cuda.get_device_name(0) if DISPOSITIVO == 'cuda' else 'sem GPU (CPU)',
    'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__,
    'scikit-learn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__,
    'matplotlib': matplotlib.__version__, 'seaborn': sns.__version__,
}
with open(os.path.join(SUB['anexos'], 'ambiente_execucao.json'), 'w', encoding='utf-8') as f:
    json.dump(ambiente, f, ensure_ascii=False, indent=2)
log.info('=' * 70)
log.info('EmoText MZ — Detecção de emoções (AfriEmo PT-MZ), configuração corrigida')
for k, v in ambiente.items():
    log.info(f'  {k}: {v}')
if not EM_COLAB and DISPOSITIVO == 'cpu' and not TESTE_RAPIDO:
    log.warning('Sem GPU: o ajuste fino completo pode demorar várias horas.')

2026-10-12 06:05:15 | INFO | ======================================================================


INFO:emotext:======================================================================


2026-10-12 06:05:15 | INFO | EmoText MZ — Detecção de emoções (AfriEmo PT-MZ), configuração corrigida


INFO:emotext:EmoText MZ — Detecção de emoções (AfriEmo PT-MZ), configuração corrigida


2026-10-12 06:05:15 | INFO |   data_execucao: 2026-10-12 06:05:15


INFO:emotext:  data_execucao: 2026-10-12 06:05:15


... (16 outputs left out)

2026-10-12 06:05:15 | INFO |   matplotlib: 3.10.0


INFO:emotext:  matplotlib: 3.10.0


2026-10-12 06:05:15 | INFO |   seaborn: 0.13.2


INFO:emotext:  seaborn: 0.13.2


## 3. Dados — AfriEmo / BRIGHTER, português de Moçambique
Os ficheiros `dev` e `test` descarregados repetem cada texto (faixas A e C); a deduplicação por texto recupera os
tamanhos oficiais (1.546 / 257 / 776). Os textos passam pela limpeza do `cleaner.py`.

In [ ]:
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'http\S+|www\.\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


DADOS = {}
for split in ['train', 'dev', 'test']:
    df = pd.read_csv(os.path.join(PASTA_DADOS, f'afriemo_ptmz_{split}.csv'), dtype={'text': str})
    df = df.drop_duplicates(subset='text').rename(columns={'text': 'texto_original'})
    df['texto'] = df['texto_original'].apply(limpar_texto)
    df[EMOCOES] = df[EMOCOES].astype(int)
    DADOS[split] = df.reset_index(drop=True)
if TESTE_RAPIDO:
    DADOS = {s: d.sample(min(len(d), 64 if s == 'train' else 40), random_state=0).reset_index(drop=True) for s, d in DADOS.items()}

Y = {s: d[EMOCOES].values for s, d in DADOS.items()}
linhas = []
for s, df in DADOS.items():
    dist = {EMO_PT[e]: int(df[e].sum()) for e in EMOCOES} | {'Neutro': int((df[EMOCOES].sum(axis=1) == 0).sum())}
    linhas.append({'particao': s, 'n': len(df), **dist})
    log.info(f'[Dados] {s:5s} n={len(df):5d} {dist}')
pd.DataFrame(linhas).to_csv(os.path.join(SUB['tabelas'], 'T_dados_afriemo.csv'), index=False, encoding='utf-8-sig')

positivos = Y['train'].sum(axis=0)
PESO_POSITIVO = np.sqrt((len(Y['train']) - positivos) / np.maximum(positivos, 1))
log.info('[Pesos] ' + ', '.join(f'{EMO_PT[e]}={p:.2f}' for e, p in zip(EMOCOES, PESO_POSITIVO)))

2026-10-12 06:05:18 | INFO | [Dados] train n= 1546 {'Raiva': 90, 'Nojo': 64, 'Medo': 109, 'Alegria': 199, 'Tristeza': 276, 'Surpresa': 100, 'Neutro': 715}


INFO:emotext:[Dados] train n= 1546 {'Raiva': 90, 'Nojo': 64, 'Medo': 109, 'Alegria': 199, 'Tristeza': 276, 'Surpresa': 100, 'Neutro': 715}


2026-10-12 06:05:18 | INFO | [Dados] dev   n=  257 {'Raiva': 15, 'Nojo': 10, 'Medo': 18, 'Alegria': 33, 'Tristeza': 46, 'Surpresa': 17, 'Neutro': 119}


INFO:emotext:[Dados] dev   n=  257 {'Raiva': 15, 'Nojo': 10, 'Medo': 18, 'Alegria': 33, 'Tristeza': 46, 'Surpresa': 17, 'Neutro': 119}


2026-10-12 06:05:18 | INFO | [Dados] test  n=  776 {'Raiva': 46, 'Nojo': 33, 'Medo': 55, 'Alegria': 100, 'Tristeza': 138, 'Surpresa': 50, 'Neutro': 358}


INFO:emotext:[Dados] test  n=  776 {'Raiva': 46, 'Nojo': 33, 'Medo': 55, 'Alegria': 100, 'Tristeza': 138, 'Surpresa': 50, 'Neutro': 358}


2026-10-12 06:05:18 | INFO | [Pesos] Raiva=4.02, Nojo=4.81, Medo=3.63, Alegria=2.60, Tristeza=2.15, Surpresa=3.80


INFO:emotext:[Pesos] Raiva=4.02, Nojo=4.81, Medo=3.63, Alegria=2.60, Tristeza=2.15, Surpresa=3.80


## 4. Métricas, calibração de limiares e registo

In [ ]:
def metricas_emocoes(Y_true, Y_pred):
    m = {'macro_f1': f1_score(Y_true, Y_pred, average='macro', zero_division=0),
         'micro_f1': f1_score(Y_true, Y_pred, average='micro', zero_division=0),
         'exactidao_exacta': accuracy_score(Y_true, Y_pred)}
    for i, e in enumerate(EMOCOES):
        m[f'f1_{e}'] = f1_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
    return m


def calibrar_limiares(Y_true, probabilidades):
    limiares = []
    for i in range(Y_true.shape[1]):
        f1s = [f1_score(Y_true[:, i], (probabilidades[:, i] >= t).astype(int), zero_division=0) for t in GRELHA_LIMIARES]
        limiares.append(float(GRELHA_LIMIARES[int(np.argmax(f1s))]))
    return np.array(limiares)


def aplicar_limiares(probabilidades, limiares):
    return (probabilidades >= limiares).astype(int)


def chave(modelo, semente):
    return f'emo_ptmz__{FICHEIRO[modelo]}__s{semente}'


def ja_executado(c):
    return os.path.exists(os.path.join(SUB['execucoes'], c + '.json'))


def registar(modelo, semente, previsto, segundos=0.0, extra=None):
    c = chave(modelo, semente)
    te = DADOS['test']
    m = metricas_emocoes(Y['test'], previsto)
    reg = {'tarefa': 'emo_ptmz', 'modelo': modelo, 'semente': semente, 'segundos_treino': round(segundos, 1), **m, **(extra or {})}
    with open(os.path.join(SUB['execucoes'], c + '.json'), 'w', encoding='utf-8') as f:
        json.dump(reg, f, ensure_ascii=False, indent=2)
    prev = te[['texto_original', 'texto']].copy()
    for i, e in enumerate(EMOCOES):
        prev[f'real_{e}'] = Y['test'][:, i]
        prev[f'prev_{e}'] = previsto[:, i]
    prev.to_csv(os.path.join(SUB['previsoes'], c + '.csv'), index=False, encoding='utf-8-sig')
    rel = classification_report(Y['test'], previsto, target_names=[EMO_PT[e] for e in EMOCOES], zero_division=0, digits=4)
    with open(os.path.join(SUB['relatorios'], c + '.txt'), 'w', encoding='utf-8') as f:
        f.write(f'Emoções — AfriEmo Português MZ | modelo: {modelo} | semente: {semente}\n')
        if extra and 'limiares' in extra:
            f.write('Limiares calibrados na validação: ' + ', '.join(f'{EMO_PT[e]}={t:.2f}' for e, t in zip(EMOCOES, extra['limiares'])) + '\n')
        f.write('\n' + rel)
    log.info(f'[Resultado] {c}: macro_f1={m["macro_f1"]:.4f} | micro_f1={m["micro_f1"]:.4f} | '
             + ' '.join(f'{EMO_PT[e]}={m["f1_" + e]:.3f}' for e in EMOCOES))

## 5. Baselines: classe maioritária, léxico NRC (português) e TF-IDF + Regressão Logística

In [ ]:
if not ja_executado(chave('Maioritária', 0)):
    registar('Maioritária', 0, np.tile((Y['train'].mean(axis=0) > 0.5).astype(int), (len(Y['test']), 1)),
             extra={'regra': 'emoção prevista se presente em mais de 50% dos textos de treino'})

if not ja_executado(chave('Léxico NRC', 0)):
    destino = os.path.join(PASTA_TEMP, 'nrc.zip')
    if not os.path.exists(destino):
        import urllib.request
        pedido = urllib.request.Request(URL_NRC, headers={'User-Agent': 'curl/8.4.0'})
        with urllib.request.urlopen(pedido, timeout=120) as resposta, open(destino, 'wb') as f:
            shutil.copyfileobj(resposta, f)
    with zipfile.ZipFile(destino) as z:
        nome = [n for n in z.namelist() if n.endswith('Portuguese-NRC-EmoLex.txt')][0]
        nrc = pd.read_csv(io.BytesIO(z.read(nome)), sep='\t')
    nrc['termo'] = nrc['Portuguese Word'].astype(str).str.lower().str.strip()
    nrc = nrc[nrc['termo'].str.fullmatch(r"[\wÀ-ÿ'-]+")].groupby('termo')[EMOCOES].max()
    lex = {t: v.values.astype(int) for t, v in nrc.iterrows()}

    def nrc_emocoes(texto):
        v = np.zeros(len(EMOCOES), dtype=int)
        for t in re.findall(r"[\wÀ-ÿ']+", texto.lower()):
            if t in lex:
                v |= lex[t]
        return v

    registar('Léxico NRC', 0, np.vstack([nrc_emocoes(x) for x in DADOS['test']['texto']]),
             extra={'termos_lexico': len(lex)})

if not ja_executado(chave('TF-IDF+RL', 0)):
    t0 = time.time()
    clf = make_pipeline(
        make_union(TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                   TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        OneVsRestClassifier(LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42)))
    clf.fit(DADOS['train']['texto'], Y['train'])
    limiares = calibrar_limiares(Y['dev'], clf.predict_proba(DADOS['dev']['texto']))
    registar('TF-IDF+RL', 0, aplicar_limiares(clf.predict_proba(DADOS['test']['texto']), limiares), time.time() - t0,
             {'limiares': limiares.tolist()})

2026-10-12 06:05:19 | INFO | [Resultado] emo_ptmz__Maioritaria__s0: macro_f1=0.0000 | micro_f1=0.0000 | Raiva=0.000 Nojo=0.000 Medo=0.000 Alegria=0.000 Tristeza=0.000 Surpresa=0.000


INFO:emotext:[Resultado] emo_ptmz__Maioritaria__s0: macro_f1=0.0000 | micro_f1=0.0000 | Raiva=0.000 Nojo=0.000 Medo=0.000 Alegria=0.000 Tristeza=0.000 Surpresa=0.000


2026-10-12 06:05:25 | INFO | [Resultado] emo_ptmz__Lexico_NRC__s0: macro_f1=0.1871 | micro_f1=0.2013 | Raiva=0.147 Nojo=0.149 Medo=0.207 Alegria=0.209 Tristeza=0.321 Surpresa=0.090


INFO:emotext:[Resultado] emo_ptmz__Lexico_NRC__s0: macro_f1=0.1871 | micro_f1=0.2013 | Raiva=0.147 Nojo=0.149 Medo=0.207 Alegria=0.209 Tristeza=0.321 Surpresa=0.090


2026-10-12 06:05:34 | INFO | [Resultado] emo_ptmz__TFIDF_RL__s0: macro_f1=0.3000 | micro_f1=0.3458 | Raiva=0.279 Nojo=0.161 Medo=0.333 Alegria=0.345 Tristeza=0.466 Surpresa=0.215


INFO:emotext:[Resultado] emo_ptmz__TFIDF_RL__s0: macro_f1=0.3000 | micro_f1=0.3458 | Raiva=0.279 Nojo=0.161 Medo=0.333 Alegria=0.345 Tristeza=0.466 Surpresa=0.215


## 6. Ajuste fino de mBERT, XLM-R e AfroXLMR (3 sementes)

In [ ]:
class RegistoTreino(TrainerCallback):
    def __init__(self, c):
        self.c = c

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            itens = ' '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in logs.items())
            log.info(f'    [{self.c}] época={state.epoch:.2f} passo={state.global_step} {itens}')


class TrainerPonderado(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        peso = torch.tensor(PESO_POSITIVO, dtype=saidas.logits.dtype, device=saidas.logits.device)
        perda = torch.nn.functional.binary_cross_entropy_with_logits(saidas.logits, rotulos.to(saidas.logits.dtype), pos_weight=peso)
        return (perda, saidas) if return_outputs else perda


class ConjuntoTexto(torch.utils.data.Dataset):
    def __init__(self, cod, rot):
        self.cod, self.rot = cod, rot

    def __len__(self):
        return len(self.rot)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.cod.items()}
        item['labels'] = self.rot[i]
        return item


def sigmoide(x):
    return 1 / (1 + np.exp(-x))


def ajustar(nome_modelo, id_modelo, semente):
    c = chave(nome_modelo, semente)
    if ja_executado(c):
        log.info(f'[Saltar] {c} já concluído')
        return
    log.info(f'[Treino] {c} | {id_modelo} | lr={HIPER["taxa_aprendizagem"]} | épocas={HIPER["epocas"]}')
    set_seed(semente)
    tok = AutoTokenizer.from_pretrained(id_modelo)
    modelo = AutoModelForSequenceClassification.from_pretrained(id_modelo, num_labels=len(EMOCOES),
                                                                problem_type='multi_label_classification')

    def preparar(split):
        cod = tok(DADOS[split]['texto'].tolist(), truncation=True, max_length=HIPER['max_comprimento'])
        return ConjuntoTexto(cod, [list(map(float, r)) for r in Y[split]])

    def calcular(ep):
        logits, y = ep
        p = sigmoide(logits)
        return {'macro_f1': f1_score(y.astype(int), aplicar_limiares(p, calibrar_limiares(y.astype(int), p)),
                                     average='macro', zero_division=0)}

    pasta = os.path.join(PASTA_TEMP, c)
    args = TrainingArguments(
        output_dir=pasta, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, save_only_model=True,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
        learning_rate=HIPER['taxa_aprendizagem'], per_device_train_batch_size=HIPER['tamanho_lote'],
        per_device_eval_batch_size=64, num_train_epochs=HIPER['epocas'], weight_decay=HIPER['decaimento_pesos'],
        warmup_ratio=HIPER['proporcao_aquecimento'], fp16=(DISPOSITIVO == 'cuda'), seed=semente, data_seed=semente,
        logging_strategy='epoch', report_to='none', disable_tqdm=not EM_COLAB, dataloader_num_workers=0)
    trainer = TrainerPonderado(model=modelo, args=args, train_dataset=preparar('train'), eval_dataset=preparar('dev'),
                               data_collator=DataCollatorWithPadding(tok), compute_metrics=calcular,
                               callbacks=[RegistoTreino(c)])
    t0 = time.time()
    trainer.train()
    segundos = time.time() - t0
    limiares = calibrar_limiares(Y['dev'], sigmoide(trainer.predict(preparar('dev')).predictions))
    previsto = aplicar_limiares(sigmoide(trainer.predict(preparar('test')).predictions), limiares)
    melhor_epoca = None
    if trainer.state.best_model_checkpoint:
        passo = int(trainer.state.best_model_checkpoint.rstrip('/').split('-')[-1])
        melhor_epoca = next((round(h['epoch']) for h in trainer.state.log_history if h.get('step') == passo and 'eval_macro_f1' in h), None)
    registar(nome_modelo, semente, previsto, segundos,
             {'id_modelo': id_modelo, 'limiares': limiares.tolist(), 'melhor_epoca': melhor_epoca})
    del trainer, modelo
    if DISPOSITIVO == 'cuda':
        torch.cuda.empty_cache()
    shutil.rmtree(pasta, ignore_errors=True)


t_inicio = time.time()
for nome_modelo, id_modelo in MODELOS.items():
    for semente in SEMENTES:
        ajustar(nome_modelo, id_modelo, semente)
log.info(f'[Treino] Concluído em {(time.time() - t_inicio) / 60:.1f} min')

2026-10-12 06:05:35 | INFO | [Treino] emo_ptmz__mBERT__s42 | google-bert/bert-base-multilingual-cased | lr=3e-05 | épocas=10


INFO:emotext:[Treino] emo_ptmz__mBERT__s42 | google-bert/bert-base-multilingual-cased | lr=3e-05 | épocas=10
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (431 outputs left out)

2026-10-12 07:12:18 | INFO | [Resultado] emo_ptmz__AfroXLMR__s44: macro_f1=0.4031 | micro_f1=0.4435 | Raiva=0.283 Nojo=0.219 Medo=0.550 Alegria=0.411 Tristeza=0.570 Surpresa=0.385


INFO:emotext:[Resultado] emo_ptmz__AfroXLMR__s44: macro_f1=0.4031 | micro_f1=0.4435 | Raiva=0.283 Nojo=0.219 Medo=0.550 Alegria=0.411 Tristeza=0.570 Surpresa=0.385


2026-10-12 07:12:18 | INFO | [Treino] Concluído em 66.7 min


INFO:emotext:[Treino] Concluído em 66.7 min


## 7. Agregação (média ± desvio-padrão das sementes)

In [ ]:
execucoes = pd.DataFrame([json.load(open(f, encoding='utf-8')) for f in glob.glob(os.path.join(SUB['execucoes'], '*.json'))])
execucoes.to_csv(os.path.join(SUB['tabelas'], 'T_emocoes_todas_execucoes.csv'), index=False, encoding='utf-8-sig')


def media_dp(s):
    return f'{s.mean():.4f} ± {s.std(ddof=1):.4f}' if len(s) > 1 else f'{s.mean():.4f}'


linhas = []
for modelo in [m for m in ORDEM_MODELOS if m in execucoes['modelo'].values]:
    g = execucoes[execucoes['modelo'] == modelo]
    linha = {'modelo': modelo, 'n_sementes': len(g)}
    for m in ['macro_f1', 'micro_f1'] + [f'f1_{e}' for e in EMOCOES]:
        linha[m] = media_dp(g[m])
        linha[m + '_media'] = round(g[m].mean(), 4)
    if 'melhor_epoca' in g and g['melhor_epoca'].notna().any():
        linha['melhor_epoca'] = ', '.join(str(int(x)) for x in g['melhor_epoca'].dropna())
    linhas.append(linha)
T_emo = pd.DataFrame(linhas)
T_emo.to_csv(os.path.join(SUB['tabelas'], 'T_emocoes_resumo_v2.csv'), index=False, encoding='utf-8-sig')

lim = execucoes.dropna(subset=['limiares']) if 'limiares' in execucoes else pd.DataFrame()
if len(lim):
    T_lim = pd.DataFrame([{'modelo': r['modelo'], 'semente': r['semente'], **{EMO_PT[e]: t for e, t in zip(EMOCOES, r['limiares'])}}
                          for _, r in lim.iterrows()]).sort_values(['modelo', 'semente'])
    T_lim.to_csv(os.path.join(SUB['tabelas'], 'T_limiares_calibrados.csv'), index=False, encoding='utf-8-sig')
tempos = execucoes[execucoes['segundos_treino'] > 0].groupby('modelo')['segundos_treino'].agg(['mean', 'sum', 'count']).round(1)
tempos.to_csv(os.path.join(SUB['tabelas'], 'T_tempos_treino_emocoes.csv'), encoding='utf-8-sig')
log.info('\n===== EMOÇÕES (configuração corrigida) =====\n' + T_emo[[c for c in T_emo.columns if not c.endswith('_media')]].to_string(index=False))

2026-10-12 07:12:18 | INFO | 
===== EMOÇÕES (configuração corrigida) =====
     modelo  n_sementes        macro_f1        micro_f1        f1_anger      f1_disgust         f1_fear          f1_joy      f1_sadness     f1_surprise melhor_epoca
Maioritária           1          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          NaN
 Léxico NRC           1          0.1871          0.2013          0.1472          0.1493          0.2071          0.2085          0.3211          0.0896          NaN
  TF-IDF+RL           1          0.3000          0.3458          0.2791          0.1613          0.3333          0.3448          0.4664          0.2154          NaN
      mBERT           3 0.3663 ± 0.0280 0.4254 ± 0.0243 0.2118 ± 0.0319 0.1508 ± 0.0523 0.4553 ± 0.0391 0.4551 ± 0.0289 0.5358 ± 0.0323 0.3887 ± 0.0426      4, 4, 3
      XLM-R           3 0.4046 ± 0.0215 0.4545 ± 0.0096 0.2609 ± 0.0409 0.2258 ± 0.0559 0.5118 ± 0.0

INFO:emotext:
===== EMOÇÕES (configuração corrigida) =====
     modelo  n_sementes        macro_f1        micro_f1        f1_anger      f1_disgust         f1_fear          f1_joy      f1_sadness     f1_surprise melhor_epoca
Maioritária           1          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          0.0000          NaN
 Léxico NRC           1          0.1871          0.2013          0.1472          0.1493          0.2071          0.2085          0.3211          0.0896          NaN
  TF-IDF+RL           1          0.3000          0.3458          0.2791          0.1613          0.3333          0.3448          0.4664          0.2154          NaN
      mBERT           3 0.3663 ± 0.0280 0.4254 ± 0.0243 0.2118 ± 0.0319 0.1508 ± 0.0523 0.4553 ± 0.0391 0.4551 ± 0.0289 0.5358 ± 0.0323 0.3887 ± 0.0426      4, 4, 3
      XLM-R           3 0.4046 ± 0.0215 0.4545 ± 0.0096 0.2609 ± 0.0409 0.2258 ± 0.0559 0.5118 ± 0.0213 0.4870 ± 0.0

## 8. Figuras: F1 por emoção e matrizes de confusão do melhor modelo

In [ ]:
plt.rcParams.update({'font.size': 10.5, 'axes.titleweight': 'bold', 'figure.facecolor': 'white'})


def virgula(x, casas=1):
    return f'{x:.{casas}f}'.replace('.', ',')


modelos_fig = [m for m in ORDEM_MODELOS if m in T_emo['modelo'].values and m != 'Maioritária']
cores = dict(zip(['Léxico NRC', 'TF-IDF+RL', 'mBERT', 'XLM-R', 'AfroXLMR'], ['#b9b8b2', '#52514e', '#eb6834', '#1baf7a', '#2a78d6']))
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(EMOCOES))
larg = 0.8 / len(modelos_fig)
for i, modelo in enumerate(modelos_fig):
    g = execucoes[execucoes['modelo'] == modelo]
    medias = [g[f'f1_{e}'].mean() * 100 for e in EMOCOES]
    erros = [g[f'f1_{e}'].std(ddof=1) * 100 if len(g) > 1 else np.nan for e in EMOCOES]
    ax.bar(x + (i - (len(modelos_fig) - 1) / 2) * larg, medias, larg * 0.92, yerr=erros, capsize=2.5,
           color=cores[modelo], label=modelo, error_kw={'elinewidth': 0.9, 'ecolor': '#0b0b0b'})
ax.set_xticks(x)
ax.set_xticklabels([EMO_PT[e] for e in EMOCOES])
ax.set_ylabel('F1 (%)')
ax.set_ylim(0, 100)
ax.yaxis.grid(True, color='#e4e3df')
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(ncol=len(modelos_fig), loc='lower center', bbox_to_anchor=(0.5, 1.0), frameon=False)
ax.set_title('F1 por emoção no AfriEmo — Português MZ', pad=30)
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'grafico_f1_por_emocao_v2.png'), dpi=200, bbox_inches='tight')
plt.close()

melhor = execucoes[~execucoes['modelo'].isin(['Maioritária'])].groupby('modelo')['macro_f1'].mean().idxmax()
ficheiros = sorted(glob.glob(os.path.join(SUB['previsoes'], f'emo_ptmz__{FICHEIRO[melhor]}__s*.csv')))
mcm = sum(multilabel_confusion_matrix(pd.read_csv(f)[[f'real_{e}' for e in EMOCOES]].values,
                                      pd.read_csv(f)[[f'prev_{e}' for e in EMOCOES]].values) for f in ficheiros) / len(ficheiros)
fig, eixos = plt.subplots(2, 3, figsize=(11, 6.6))
for ax, e, m in zip(eixos.ravel(), EMOCOES, mcm):
    sns.heatmap(m, annot=np.vectorize(lambda v: virgula(v, 0 if len(ficheiros) == 1 else 1))(m), fmt='', cmap='Blues',
                cbar=False, xticklabels=['Ausente', 'Presente'], yticklabels=['Ausente', 'Presente'],
                linewidths=2, linecolor='white', ax=ax)
    ax.set_title(EMO_PT[e], fontsize=11)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
fig.suptitle(f'Matrizes de confusão por emoção — {melhor}', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SUB['figuras'], 'figura_matrizes_confusao_emocoes_v2.png'), dpi=200, bbox_inches='tight')
plt.close()
log.info(f'[Figuras] melhor modelo: {melhor}; ' + ', '.join(sorted(os.listdir(SUB['figuras']))))

2026-10-12 07:12:20 | INFO | [Figuras] melhor modelo: XLM-R; figura_matrizes_confusao_emocoes_v2.png, grafico_f1_por_emocao_v2.png


INFO:emotext:[Figuras] melhor modelo: XLM-R; figura_matrizes_confusao_emocoes_v2.png, grafico_f1_por_emocao_v2.png


## 9. Material para os anexos e exportação

In [ ]:
config = {'hiperparametros': HIPER, 'pesos_positivos': dict(zip([EMO_PT[e] for e in EMOCOES], np.round(PESO_POSITIVO, 4).tolist())),
          'sementes': SEMENTES, 'modelos': MODELOS, 'ambiente': ambiente,
          'motivo': 'A configuração inicial (5 épocas, lr 2e-5, limiar fixo 0,5, perda não ponderada) levou os modelos a '
                    'prever sistematicamente a ausência de emoções; adoptou-se perda ponderada e calibração de limiares na validação.'}
with open(os.path.join(SUB['anexos'], 'Anexo_configuracao_emocoes_v2.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

with open(os.path.join(SUB['anexos'], 'Anexo_relatorios_emocoes_v2.txt'), 'w', encoding='utf-8') as f:
    for modelo in [m for m in ORDEM_MODELOS if m in execucoes['modelo'].values]:
        caminho = os.path.join(SUB['relatorios'], chave(modelo, 0 if modelo in BASELINES else SEMENTES[0]) + '.txt')
        if os.path.exists(caminho):
            f.write(open(caminho, encoding='utf-8').read() + '\n' + '-' * 70 + '\n')

p = pd.read_csv(os.path.join(SUB['previsoes'], chave(melhor, 0 if melhor in BASELINES else SEMENTES[0]) + '.csv'))
reais, prevs = p[[f'real_{e}' for e in EMOCOES]].values, p[[f'prev_{e}' for e in EMOCOES]].values
nomes = lambda v: ', '.join(EMO_PT[e] for e, b in zip(EMOCOES, v) if b) or 'Neutro'
exemplos = []
for acerto in [True, False]:
    idx = np.where((reais == prevs).all(axis=1) == acerto)[0]
    for i in pd.Series(idx).sample(min(10, len(idx)), random_state=42):
        exemplos.append({'modelo': melhor, 'texto': p.loc[i, 'texto'], 'real': nomes(reais[i]), 'previsto': nomes(prevs[i]),
                         'correcto': 'Sim' if acerto else 'Não'})
pd.DataFrame(exemplos).to_csv(os.path.join(SUB['anexos'], 'Anexo_exemplos_emocoes_v2.csv'), index=False, encoding='utf-8-sig')

log.info('[Fim] Avaliação de emoções concluída.')
for h in log.handlers:
    h.flush()
with open(os.path.join(SUB['anexos'], 'Anexo_log_emocoes_v2.log'), 'w', encoding='utf-8') as destino:
    for caminho in sorted(glob.glob(os.path.join(SUB['logs'], 'execucao_*.log'))):
        destino.write(open(caminho, encoding='utf-8').read())

zip_destino = os.path.join(PASTA_BASE, 'resultados_teste' if TESTE_RAPIDO else 'resultados_emocoes_v2')
shutil.make_archive(zip_destino, 'zip', PASTA_RES)
print('Ficheiro de resultados:', zip_destino + '.zip')
if EM_COLAB:
    from google.colab import files
    files.download(zip_destino + '.zip')

2026-10-12 07:12:20 | INFO | [Fim] Avaliação de emoções concluída.


INFO:emotext:[Fim] Avaliação de emoções concluída.


Ficheiro de resultados: /content/drive/MyDrive/EmoTextColab/resultados_emocoes_v2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>